# 01 · Snapshot check — nra-snapshot-em (Version 1)

First read of the frozen corpus behind the New-Realism Analyzer
(github.com/computational-ontology/model). The dataset holds every
Constitute section tagged `em` (Emergency provisions) in in-force
constitutions, dumped 2026-09-14: 740 English and 180 Spanish sections,
plus two hash-only manifests. This notebook trains nothing; it verifies
that the snapshot is complete and describes its shape.

Rules: notebooks read only this dataset, never the Constitute API;
the dataset is private (CC BY-NC texts); every notebook names the
dataset version it read (here: Version 1).

## 0 · Where Kaggle mounted the dataset

Kaggle currently mounts datasets under `/kaggle/input/datasets/<owner>/<slug>/`,
not directly under `/kaggle/input/<slug>/` as older tutorials show. This cell walks
`/kaggle/input` and prints every folder with its file count, so the real path is
recorded in the notebook rather than assumed.

In [ ]:
import os
for dirpath, dirnames, filenames in os.walk("/kaggle/input"):
    print(dirpath, "→", len(filenames), "files")

## 1 · What is in the mounted folder

`root` is the path found above. We expect 922 files: 920 section texts named
`<constitution_id>__<section_id>__<lang>.txt` and the two manifests
`snapshot_manifest_en.json` / `snapshot_manifest_es.json`.

In [ ]:
root = "/kaggle/input/datasets/luisdscientist/nra-snapshot-em"
files = sorted(os.listdir(root))
print(len(files), "files")
print([f for f in files if f.endswith(".json")])
print(files[:5])

## 2 · Load the manifests

The manifests are the *unamendable record* of the corpus: one row per section with
identifiers, the chapter/article breadcrumb, language, copyright and translator
metadata, length and SHA-256 — but no text. Both languages are concatenated into
one DataFrame (920 rows).

In [ ]:
import json, pandas as pd
frames = []
for lang in ["en", "es"]:
    with open(f"{root}/snapshot_manifest_{lang}.json", encoding="utf-8") as f:
        m = json.load(f)
    frames.append(pd.DataFrame(m["sections"]))
df = pd.concat(frames, ignore_index=True)
print(df.shape)
df.head()

## 3 · Q1 — How are the sections spread across constitutions?

Counts per constitution and language, plus summary statistics. A skewed
distribution (a few constitutions with 20–28 sections, most with 2–5) matters for
annotation: the campaign covers the whole population, so heavy constitutions
dominate unless the pilot is stratified.

In [ ]:
per_cons = df.groupby(["lang", "constitution_id"]).size().sort_values(ascending=False)
print(per_cons.head(10))
print(per_cons.groupby(level="lang").describe())

## 4 · Q2 — How long is a section?

Character counts per language with the 50th, 90th and 99th percentiles, and a
log-scale histogram. Short sections (median ≈ 300 characters) fit inside encoder
context windows and keep the annotation cost per unit low; the long tail is where
enumerated lists of suspendable rights live.

In [ ]:
print(df.groupby("lang")["n_chars"].describe(percentiles=[.5, .9, .99]))
df["n_chars"].plot(kind="hist", bins=60, logy=True, title="Section length (chars)");

## 5 · Q3 — Where do constitutions put their emergency clauses?

The first breadcrumb element is the chapter. Whether a clause sits in a dedicated
"State of Emergency" chapter, among the executive's powers, or inside the rights
catalogue is a structural signal about how the emergency is framed — a descriptive
first look at what the operator task (↓ naturalised / ✦ revealed) will later annotate.

Raw headings first: these counts are per *heading string*, so a single constitution
with a 23-section emergency chapter dominates, and the same chapter type appears
under dozens of spellings and languages.

In [ ]:
chapter = df["header"].str.split(">").str[0].str.strip()
print(chapter.value_counts().head(15))

### 5b · Chapter types, counted per constitution

A keyword classifier (v2) maps headings to a few chapter *types*, and counts
constitutions rather than sections. Rule order matters: the emergency test runs
first so that "Guarantees of the Constitution and the Emergency" is not captured
by the rights rule. The "other" sample at the end shows what the rules still miss.

In [ ]:
import re

def chapter_type(h):
    h = (h or "").lower()
    if not h.strip():
        return "no heading"
    if re.search(r"emergenc|exception|siege|sitio|defen[cs]|excepci|urgenc|extraordinar|\bwar\b|guerra|crisis|martial", h):
        return "emergency/defence chapter"
    if re.search(r"right|freedom|derecho|libertad|garant|individual|citizen|ciudadan|persona|dignit", h):
        return "rights catalogue"
    if re.search(r"parliament|legislat|assembly|congres|senado|c[aá]mara|national council|diet|majlis|jirga", h):
        return "legislature"
    if re.search(r"judic|court|tribunal|review|revision|reform|amend|enmienda", h):
        return "judiciary / review / amendment"
    if re.search(r"president|executive|government|ejecutivo|gobierno|power|poder|instrument|authorit|state power|organi[sz]ation of the state|branches|king|crown|monarch|head of state|cabinet|minister|administration", h):
        return "executive/organisation of powers"
    if re.search(r"general|final|miscellan|transitional|transitor|provisions", h):
        return "general / final provisions"
    return "other"

In [ ]:
df["chapter_type"] = chapter.map(chapter_type)
by_cons = df.groupby("chapter_type")["constitution_id"].nunique().sort_values(ascending=False)
print("constitutions with >=1 em section in each chapter type:"); print(by_cons)
print("\nsections per chapter type:"); print(df["chapter_type"].value_counts())
print("\nshare still 'other':", round((df["chapter_type"] == "other").mean() * 100, 1), "%")
print("\nsample of 'other' headings:")
print(chapter[df["chapter_type"] == "other"].drop_duplicates().head(25).to_string())

## 6 · What this notebook establishes

- Mount path on Kaggle: `/kaggle/input/datasets/luisdscientist/nra-snapshot-em` (dataset Version 1, 922 files).
- The snapshot is complete and readable from the manifests alone: 920 sections, SHA-256 per section.
- Population for campaign 1: 740 EN / 180 ES sections; 175 / 47 constitutions; median 3 sections per constitution (max 28, Germany); median length 295 characters, 90th percentile ≈ 935, one 12 266-character outlier.
- Chapter types (keyword classifier v2, counted as constitutions with ≥1 `em` section): executive/organisation of powers 95 · rights catalogue 86 · legislature 55 · emergency/defence chapter 33 · judiciary/review/amendment 25 · general/final provisions 20 · unclassified 65 (18.4 % of sections; mostly uninformative headings such as "Part 1", plus a state-structure family still to add).
- First empirical observation: emergency provisions appear far more often as limitation clauses inside the rights catalogue (86 constitutions) or among executive powers (95) than in a dedicated emergency chapter (33). Whether the emergency is framed as "where rights are suspended" or as "a regime with its own rules" is exactly the contrast the operator task (↓ naturalised / ✦ revealed) will annotate.
- Proposed sampling stratum for the 100-section pilot: `chapter_type`.

Next: `02_stage1_llm_bakeoff.ipynb` — open-weights LLM with a JSON schema on 50 of these sections. This notebook is exported to the repository as `notebooks/01_snapshot_check.ipynb`.